TP FINAL — Du dataset au modèle déployé 

PARTIE A — Chargement & choix des variables

ÉTAPE 1 Charger le fichier Excel et le convertir en CSV 

In [2]:
import pandas as pd, numpy as np 
import warnings; warnings.filterwarnings('ignore')

In [12]:
df = pd.read_excel('../data/dataset_assurance_ML.xlsx') 
df.to_csv('../data/dataset_assurance_ML.csv', index=False, encoding='utf_8') 

In [14]:
df = pd.read_csv('../data/dataset_assurance_ML.csv', encoding='utf_8')
print(df.shape) 

(500, 27)


 ÉTAPE 2   La variable cible 

In [16]:
TARGET = 'Résiliation'
print(df[TARGET].value_counts())
print(df[TARGET].value_counts(normalize='utf_8').round(2)) 

Résiliation
0    450
1     50
Name: count, dtype: int64
Résiliation
0    0.9
1    0.1
Name: proportion, dtype: float64


ÉTAPE 3   Détecter une fuite de données (data leakage) 

In [17]:
print(pd.crosstab(df['Statut Contrat'], df[TARGET])) 

Résiliation       0   1
Statut Contrat         
Actif           450   0
Résilié           0  36
Suspendu          0  14


 ÉTAPE 4   Choisir les variables numériques et catégorielles 

In [18]:
num_cols = ['Âge', 'Salaire Annuel (€)', 'Prime Annuelle (€)', 'Ancienneté (mois)',
            'Coeff. Bonus-Malus', 'Nb Sinistres (3 ans)', 
            'Montant Sinistres (€)', 'Score Risque (0-100)'] 

In [22]:
cat_cols = ['Type Contrat', 'Catégorie Prof.', 'Usage Véhicule', 'Dernier Sinistre']
X = df[num_cols + cat_cols]
y = df[TARGET]
print(X.shape, y.shape) 

(500, 12) (500,)


ÉTAPE 5   Vérifier le lien avec la cible 

In [23]:
print(X[num_cols].corrwith(y).round(3).sort_values(ascending=False))
print(df.groupby('Dernier Sinistre')[TARGET].mean().round(2).sort_values()) 

Nb Sinistres (3 ans)     0.455
Score Risque (0-100)     0.441
Coeff. Bonus-Malus       0.412
Montant Sinistres (€)    0.284
Prime Annuelle (€)       0.041
Âge                     -0.006
Salaire Annuel (€)      -0.012
Ancienneté (mois)       -0.055
dtype: float64
Dernier Sinistre
Aucun                    0.03
Incendie                 0.16
Catastrophe naturelle    0.18
Accident                 0.26
Bris de glace            0.30
Vol                      0.30
Dégât des eaux           0.36
Name: Résiliation, dtype: float64


PARTIE B — Pipeline, entraînement & évaluation 

 ÉTAPE 6   Séparer train / test 

In [24]:
from sklearn.model_selection import train_test_split 
X_train, X_test, y_train, y_test = train_test_split( 
                                                    X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)
print(y_train.mean().round(2), y_test.mean().round(2)) 

(400, 12) (100, 12)
0.1 0.1


 ÉTAPE 7   Le prétraitement en un seul objet : ColumnTransformer 

In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

 ÉTAPE 8   Deux candidats dans un Pipeline 

In [26]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
candidats = {
    'Régression Logistique': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, max_depth=4, min_samples_leaf=10,
        class_weight='balanced', random_state=42),
    }
pipelines = {nom: Pipeline([('prep', preprocessor), ('model', algo)])
             for nom, algo in candidats.items()} 

 ÉTAPE 9   Comparer par validation croisée 

In [27]:
from sklearn.model_selection import cross_val_score
for nom, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc')
    print(f'{nom:22s} AUC = {scores.mean():.3f} ± {scores.std():.3f}') 

Régression Logistique  AUC = 0.745 ± 0.113
Random Forest          AUC = 0.801 ± 0.096


ÉTAPE 10   Entraîner le modèle retenu et évaluer sur le test 

In [30]:
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)
pipeline = pipelines['Random Forest']
pipeline.fit(X_train, y_train)
y_pred  = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1] 
print('Accuracy :', accuracy_score(y_test, y_pred), (3))
print('F1       :', f1_score(y_test, y_pred), (3))
print('ROC-AUC  :', roc_auc_score(y_test, y_proba), (3))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Reste', 'Résilie'])) 

Accuracy : 0.87 3
F1       : 0.5185185185185185 3
ROC-AUC  : 0.8533333333333333 3
[[80 10]
 [ 3  7]]
              precision    recall  f1-score   support

       Reste       0.96      0.89      0.92        90
     Résilie       0.41      0.70      0.52        10

    accuracy                           0.87       100
   macro avg       0.69      0.79      0.72       100
weighted avg       0.91      0.87      0.88       100



PARTIE C — Sauvegarde & interrogation du modèl

 ÉTAPE 12   Sauvegarder le pipeline complet 

In [32]:
import joblib, os
os.makedirs('../models', exist_ok=True)
joblib.dump(pipeline, '../models/pipeline_resiliation.pkl')
print(os.path.getsize('../models/pipeline_resiliation.pkl') / 1024, 'Ko') 

493.705078125 Ko


 ÉTAPE 13   Sauvegarder les métadonnées pour l'interface 

In [33]:
import json
meta = { 
        'modele': 'Random Forest',
        'auc_test': round(float(roc_auc_score(y_test, y_proba)), 3),
        'num_cols': num_cols,
        'cat_cols': cat_cols,
        'num_ranges': {c: {'min': float(X[c].min()), 'max': float(X[c].max()),
                           'median': float(X[c].median())} for c in num_cols},
        'cat_values': {c: sorted(X[c].unique().tolist()) for c in cat_cols},
        }
with open('../models/metadata.json', 'w', encoding='utf_8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
    meta 

 ÉTAPE 14   Interroger le modèle sur un nouveau client 

In [35]:
modele = joblib.load('../models/pipeline_resiliation.pkl')
client_risque = pd.DataFrame([{
    'Âge': 34, 'Salaire Annuel (€)': 28000, 'Prime Annuelle (€)': 950,
    'Ancienneté (mois)': 6, 'Coeff. Bonus-Malus': 1.25, 'Nb Sinistres (3 ans)': 3,
    'Montant Sinistres (€)': 4200, 'Score Risque (0-100)': 72,
    'Type Contrat': 'Bronze', 'Catégorie Prof.': 'Entrepreneur',
    'Usage Véhicule': 'Professionnel', 'Dernier Sinistre': 'Vol',
    }])
print('Classe :', modele.predict(client_risque))
print('Proba  :', modele.predict_proba(client_risque)[0, 1].round(3)) 

Classe : [1]
Proba  : 0.816


 ÉTAPE 15   Provoquer l'erreur classique, puis passer en script 

In [36]:
try:
    modele.predict(client_risque.drop(columns=['Score Risque (0-100)']))
except Exception as e:
    print('ERREUR :', e)  

ERREUR : columns are missing: {'Score Risque (0-100)'}
